# Exploring RAG Chatbot

This notebook provides an interactive environment to explore the RAG chatbot functionality.

## Setup

First, let's import the necessary modules and load configuration.

In [ ]:
import sys
from pathlib import Path

# Add src to path
sys.path.insert(0, str(Path('../src').resolve()))

from config import Config
from document_processor import DocumentProcessor
from embeddings import EmbeddingGenerator
from vector_store import VectorStore
from rag_agent import RAGChatbot

## 1. Process Documents

Let's start by processing documents from the knowledge base.

In [ ]:
# Initialize document processor
processor = DocumentProcessor(chunk_size=500, chunk_overlap=50)

# Process documents
knowledge_base_path = Path('../data/knowledge_base')
chunks = processor.process_directory(str(knowledge_base_path))

print(f"Processed {len(chunks)} chunks")
print(f"\nSample chunk:")
print(f"Content: {chunks[0].content[:200]}...")
print(f"Metadata: {chunks[0].metadata}")

## 2. Generate Embeddings

Note: This requires Azure OpenAI to be configured.

In [ ]:
# Initialize embedding generator
embedding_gen = EmbeddingGenerator(
    azure_openai_endpoint=Config.AZURE_OPENAI_ENDPOINT,
    azure_openai_api_key=Config.AZURE_OPENAI_API_KEY,
    embedding_deployment=Config.AZURE_OPENAI_EMBEDDING_DEPLOYMENT
)

print(f"Embedding dimension: {embedding_gen.get_embedding_dimension()}")

## 3. Store in Vector Database

Store the chunks and embeddings in Azure AI Search.

In [ ]:
# Initialize vector store
vector_store = VectorStore(
    search_endpoint=Config.AZURE_SEARCH_ENDPOINT,
    search_api_key=Config.AZURE_SEARCH_API_KEY,
    index_name=Config.AZURE_SEARCH_INDEX_NAME
)

# Create index
await vector_store.create_index()

print("Vector store initialized")

## 4. Test RAG Chatbot

Initialize and test the RAG chatbot.

In [ ]:
# Initialize chatbot
chatbot = RAGChatbot(
    azure_openai_endpoint=Config.AZURE_OPENAI_ENDPOINT,
    azure_openai_api_key=Config.AZURE_OPENAI_API_KEY,
    chat_deployment=Config.AZURE_OPENAI_CHAT_DEPLOYMENT,
    embedding_deployment=Config.AZURE_OPENAI_EMBEDDING_DEPLOYMENT,
    vector_store=vector_store,
    top_k=Config.TOP_K_RESULTS
)

print("Chatbot initialized")

In [ ]:
# Test with a question
question = "What products does TechCorp offer?"
response = await chatbot.chat(question)

print(f"Question: {question}")
print(f"\nResponse: {response.response}")
print(f"\nSources: {response.sources}")

## 5. Interactive Chat

Try asking multiple questions to test the chatbot.

In [ ]:
# Interactive loop
questions = [
    "What are your office hours?",
    "What is your return policy?",
    "How can I contact support?"
]

for question in questions:
    response = await chatbot.chat(question)
    print(f"\nQ: {question}")
    print(f"A: {response.response}")
    print(f"Sources: {response.sources}")

## 6. Conversation History

View the conversation history.

In [ ]:
history = chatbot.get_history()

print(f"Conversation history ({len(history)} messages):\n")
for i, msg in enumerate(history):
    print(f"{i+1}. [{msg.role}]: {msg.content[:100]}...")